# Day 9 v2 — Silver: All Report Types (For Loop Pattern)

**What changed from v1:**
- Processes all 3 report types: `kpi_report`, `sla_report`, `state_breakdown`
- Wrapped in a for loop using report config list
- Each report type has its own `flatten_fn` (schema varies per report)
- Full overwrite (MERGE is v3)

## Report type schemas

### kpi_report
```json
{ "report_period": "2026-06", "summary": { "total_sessions": ..., "total_energy_kwh": ..., ... } }
```

### sla_report
```json
{ "report_period": "2026-06", "sla_metrics": { "uptime_pct": 99.7, "avg_response_time_sec": 2.1,
  "incidents": 3, "mttr_hours": 1.5, "chargers_meeting_sla": 145, "total_chargers": 147 } }
```

### state_breakdown
```json
{ "report_period": "2026-06",
  "states": [
    { "state_code": "NSW", "sessions": 5526, "energy_kwh": 277350, "revenue_aud": 138555 },
    { "state_code": "VIC", "sessions": 4236, "energy_kwh": 212580, "revenue_aud": 106290 }, ...
  ]
}
```
State breakdown is exploded — one row per state per month.

In [ ]:
# Cell 1 — Imports
from pyspark.sql import functions as F
from datetime import datetime, timezone

print("Imports OK")

In [ ]:
# Cell 2 — Parameters (hardcoded for learning)
LOAD_TYPE       = "incremental"
INGESTION_YEAR  = "2026"
INGESTION_MONTH = "06"

BRONZE_REPORTS = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/reports"
SILVER_REPORTS = "/Volumes/dbw_ev_intelligence_dev/default/silver-volume/reports"

YYYYMM = f"{INGESTION_YEAR}{INGESTION_MONTH}"
PARTITION_PATH = f"{BRONZE_REPORTS}/{INGESTION_YEAR}/{INGESTION_MONTH}"

RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

print(f"Bronze partition : {PARTITION_PATH}")
print(f"Silver base      : {SILVER_REPORTS}")
print(f"RUN_TS           : {RUN_TS}")

In [ ]:
# Cell 3 — Flatten functions per report type
# Each function takes the raw Spark DataFrame and returns a flat DataFrame.

def flatten_kpi(raw_df, report_year, report_month, run_ts, load_type):
    return raw_df.select(
        F.col("report_period").cast("string").alias("report_period"),
        F.to_timestamp(F.col("generated_at")).alias("generated_at"),
        F.col("summary.total_sessions").cast("long").alias("total_sessions"),
        F.col("summary.total_energy_kwh").cast("decimal(14,2)").alias("total_energy_kwh"),
        F.col("summary.total_revenue_aud").cast("decimal(14,2)").alias("total_revenue_aud"),
        F.col("summary.avg_session_duration_min").cast("decimal(8,2)").alias("avg_session_duration_min"),
        F.col("summary.avg_energy_per_session_kwh").cast("decimal(8,2)").alias("avg_energy_per_session_kwh"),
        F.lit(int(report_year)).alias("report_year"),
        F.lit(int(report_month)).alias("report_month"),
        F.lit(run_ts).cast("timestamp").alias("silver_ingested_at"),
        F.lit(load_type).alias("silver_load_type"),
        F.lit("pl_silver_reports_v2").alias("silver_pipeline"),
    )


def flatten_sla(raw_df, report_year, report_month, run_ts, load_type):
    return raw_df.select(
        F.col("report_period").cast("string").alias("report_period"),
        F.to_timestamp(F.col("generated_at")).alias("generated_at"),
        F.col("sla_metrics.uptime_pct").cast("decimal(6,3)").alias("uptime_pct"),
        F.col("sla_metrics.avg_response_time_sec").cast("decimal(8,2)").alias("avg_response_time_sec"),
        F.col("sla_metrics.incidents").cast("integer").alias("incidents"),
        F.col("sla_metrics.mttr_hours").cast("decimal(8,2)").alias("mttr_hours"),
        F.col("sla_metrics.chargers_meeting_sla").cast("integer").alias("chargers_meeting_sla"),
        F.col("sla_metrics.total_chargers").cast("integer").alias("total_chargers"),
        F.lit(int(report_year)).alias("report_year"),
        F.lit(int(report_month)).alias("report_month"),
        F.lit(run_ts).cast("timestamp").alias("silver_ingested_at"),
        F.lit(load_type).alias("silver_load_type"),
        F.lit("pl_silver_reports_v2").alias("silver_pipeline"),
    )


def flatten_state_breakdown(raw_df, report_year, report_month, run_ts, load_type):
    # Explode the states array — one row per state per month
    exploded = raw_df.select(
        F.col("report_period").cast("string").alias("report_period"),
        F.to_timestamp(F.col("generated_at")).alias("generated_at"),
        F.explode(F.col("states")).alias("s")
    )
    return exploded.select(
        "report_period",
        "generated_at",
        F.col("s.state_code").cast("string").alias("state_code"),
        F.col("s.sessions").cast("long").alias("sessions"),
        F.col("s.energy_kwh").cast("decimal(14,2)").alias("energy_kwh"),
        F.col("s.revenue_aud").cast("decimal(14,2)").alias("revenue_aud"),
        F.lit(int(report_year)).alias("report_year"),
        F.lit(int(report_month)).alias("report_month"),
        F.lit(run_ts).cast("timestamp").alias("silver_ingested_at"),
        F.lit(load_type).alias("silver_load_type"),
        F.lit("pl_silver_reports_v2").alias("silver_pipeline"),
    )


print("Flatten functions defined.")

In [ ]:
# Cell 4 — Report config list
# file_pattern: filename under the monthly partition folder
# flatten_fn  : function to apply to the raw DataFrame
# silver_path : sub-path under SILVER_REPORTS

REPORTS = [
    {
        "report_type"  : "kpi_report",
        "file_pattern" : f"kpi_report_{YYYYMM}.json",
        "flatten_fn"   : flatten_kpi,
        "silver_path"  : f"{SILVER_REPORTS}/kpi_report",
        "natural_key"  : "report_period",
    },
    {
        "report_type"  : "sla_report",
        "file_pattern" : f"sla_report_{YYYYMM}.json",
        "flatten_fn"   : flatten_sla,
        "silver_path"  : f"{SILVER_REPORTS}/sla_report",
        "natural_key"  : "report_period",
    },
    {
        "report_type"  : "state_breakdown",
        "file_pattern" : f"state_breakdown_{YYYYMM}.json",
        "flatten_fn"   : flatten_state_breakdown,
        "silver_path"  : f"{SILVER_REPORTS}/state_breakdown",
        "natural_key"  : ["report_period", "state_code"],  # composite key
    },
]

print(f"Reports to process: {len(REPORTS)}")
for r in REPORTS:
    print(f"  {r['report_type']:<20} file={r['file_pattern']}")

In [ ]:
# Cell 5 — For loop: transform all report types

results = []

for report in REPORTS:
    rtype     = report["report_type"]
    file_path = f"{PARTITION_PATH}/{report['file_pattern']}"
    silver_p  = report["silver_path"]

    print(f"Processing: {rtype} ...", end=" ")

    try:
        # Step 1: Read Bronze JSON
        raw_df = (
            spark.read
            .option("multiLine", True)
            .json(file_path)
        )
        bronze_rows = raw_df.count()

        # Step 2: Flatten using report-specific function
        flat_df = report["flatten_fn"](
            raw_df, INGESTION_YEAR, INGESTION_MONTH, RUN_TS, LOAD_TYPE
        )
        silver_rows = flat_df.count()

        # Step 3: Write to Silver Delta (full overwrite)
        (
            flat_df.write.format("delta")
            .mode("overwrite").option("overwriteSchema", "true")
            .save(silver_p)
        )

        print(f"OK  (bronze={bronze_rows} -> silver={silver_rows})")
        results.append({"report": rtype, "status": "succeeded",
                        "bronze_rows": bronze_rows, "silver_rows": silver_rows, "error": None})

    except Exception as ex:
        print(f"FAILED")
        results.append({"report": rtype, "status": "failed",
                        "bronze_rows": 0, "silver_rows": 0, "error": str(ex)})

print("\nAll reports done.")

In [ ]:
# Cell 6 — Run summary

succeeded = [r for r in results if r["status"] == "succeeded"]
failed    = [r for r in results if r["status"] == "failed"]

print("=" * 65)
print("SILVER BLOB REPORTS v2 — RUN SUMMARY")
print("=" * 65)
print(f"  run_timestamp : {RUN_TS}")
print(f"  succeeded     : {len(succeeded)}")
print(f"  failed        : {len(failed)}")
print()

for r in results:
    tag = "[OK]  " if r["status"] == "succeeded" else "[FAIL]"
    print(f"  {tag} {r['report']:<25} bronze={r['bronze_rows']:>4}  silver={r['silver_rows']:>4}")
    if r["error"]:
        print(f"         Error: {r['error']}")

if failed:
    raise Exception(f"{len(failed)} report(s) failed — check output above.")
else:
    print(f"\nAll {len(succeeded)} reports written to Silver successfully.")